<a href="https://colab.research.google.com/github/listeven06/msd-hit-prediction/blob/main/Final_Project_Model_Building_STATS426_Ethan_Biddle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
from google.colab import files
uploaded = files.upload()

Saving audio_features.csv to audio_features (1).csv
Saving leader_board.csv to leader_board (1).csv


In [29]:
# import libraries
import pandas as pd
import numpy as np
import ast
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# load the datasets into pandas DataFrames
audio = pd.read_csv("audio_features.csv")
lb = pd.read_csv("leader_board.csv")

# check they loaded correctly
print(audio.head())
print(lb.head())

   index                                             SongID  \
0      0     -twistin'-White Silver SandsBill Black's Combo   
1      1  ¿Dònde Està Santa Claus? (Where Is Santa Claus...   
2      2             ......And Roses And RosesAndy Williams   
3      3           ...And Then There Were DrumsSandy Nelson   
4      4                ...Baby One More TimeBritney Spears   

            Performer                                              Song  \
0  Bill Black's Combo                      -twistin'-White Silver Sands   
1          Augie Rios  ¿Dònde Està Santa Claus? (Where Is Santa Claus?)   
2       Andy Williams                         ......And Roses And Roses   
3        Sandy Nelson                      ...And Then There Were Drums   
4      Britney Spears                             ...Baby One More Time   

                                       spotify_genre        spotify_track_id  \
0                                                 []                     NaN   
1         

In [30]:
# Convert WeekID to datetime
lb["WeekID"] = pd.to_datetime(lb["WeekID"], format="%m/%d/%Y", errors="coerce")

# Aggregate weekly chart data to the song level
lb_song = (
    lb.groupby("SongID", as_index=False)
      .agg(
          # First and last week the song appeared on the chart
          first_week=("WeekID", "min"),
          last_week=("WeekID", "max"),

          # Best (highest) chart position achieved
          peak_position=("Peak Position", "min"),

          # Total number of weeks the song appeared on the Billboardd Hot 100 chart.
          weeks_on_chart=("Weeks on Chart", "max"),
      )
      .assign(
          # Defining the hit:
          # Peak Position <= 10
          # Weeks on the chart >= 15
          hit=lambda d: ((d.peak_position <= 10) & (d.weeks_on_chart >= 15)).astype(int)
      )
)

In [31]:
# select only audio features relevant for analysis and modeling
relevant_audio_features = [
    "SongID",
    "spotify_genre",
    "spotify_track_duration_ms",
    "spotify_track_explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"
]

# subset the uncleaned dataset to include only the selected features
audio = audio[relevant_audio_features]

# remove rows with missing values to ensure a complete feature set
audio = audio.dropna()

In [32]:
# convert boolean to 0 and 1
audio["spotify_track_explicit"] = audio["spotify_track_explicit"].astype(int)

In [33]:
# replace empty [] genre values with ["unknown"] for consistency
def normalize_genres(x):
    if x == "[]":
        return ["unknown"]
    return ast.literal_eval(x)

# apply to audio df
audio["spotify_genre"] = audio["spotify_genre"].apply(normalize_genres)

In [34]:
# Collect unique genres
all_genres = sorted({g for genres in audio["spotify_genre"] for g in genres})

# Reserve special tokens for
# 1) padding
# 2) and unknown/new genres in unseen songs (testing)
PAD = "<PAD>"
UNK = "<UNK>"

# initialize a dictionary mapping of genres, starting from 2
genre2idx = {PAD: 0, UNK: 1}
for g in all_genres:
    if g not in genre2idx:
        genre2idx[g] = len(genre2idx)

# reverse mapping, from index to genre string.
idx2genre = {i: g for g, i in genre2idx.items()}

# counts the number of unique genres
vocab_size = len(genre2idx)

print(
    f"Genre vocabulary size: {vocab_size:,}\n"
    f"Example genres: {', '.join(list(genre2idx.keys())[2:7])}"
)

Genre vocabulary size: 1,045
Example genres: a cappella, acid house, acid jazz, acoustic blues, acoustic pop


In [35]:
# set limit on the number of genres per song
MAX_GENRES = 5

# function to encode string genre values to integer ID
def encode_genres(genres):
    # get ids from genre2idx, falls back to <unk> if not seen.
    ids = [genre2idx.get(g, genre2idx[UNK]) for g in genres]

    # truncate
    ids = ids[:MAX_GENRES]
    # pad with 0 if less than 5 genres
    ids += [genre2idx[PAD]] * (MAX_GENRES - len(ids))

    return ids

# apply to audio dataset
audio["spotify_genre_ids"] = audio["spotify_genre"].apply(encode_genres)

audio.head(5)

,SongID,spotify_genre,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,spotify_genre_ids
2,......And Roses And RosesAndy Williams,"[adult standards, brill building pop, easy lis...",166106.0,0,0.154,0.185,5.0,-14.063,1.0,0.0315,0.91100,0.000267,0.112,0.150,83.969,4.0,"[8, 127, 365, 614, 0]"
3,...And Then There Were DrumsSandy Nelson,"[rock-and-roll, space age pop, surf music]",172066.0,0,0.588,0.672,11.0,-17.278,0.0,0.0361,0.00256,0.745000,0.145,0.801,121.962,4.0,"[844, 910, 926, 0, 0]"
4,...Baby One More TimeBritney Spears,"[dance pop, pop, post-teen pop]",211066.0,0,0.759,0.699,0.0,-5.745,0.0,0.0307,0.20200,0.000131,0.443,0.907,92.960,4.0,"[277, 764, 785, 0, 0]"
5,...Ready For It?Taylor Swift,"[pop, post-teen pop]",208186.0,0,0.613,0.764,2.0,-6.509,1.0,0.1360,0.05270,0.000000,0.197,0.417,160.015,4.0,"[764, 785, 0, 0, 0]"
7,'65 Love AffairPaul Davis,"[album rock, bubblegum pop, country rock, folk...",219813.0,0,0.647,0.686,2.0,-4.247,0.0,0.0274,0.43200,0.000006,0.133,0.952,155.697,4.0,"[16, 145, 268, 409, 614]"


In [36]:
# Convert the encoded genre-id lists in the dataframe to a (N, MAX_GENRES) tensor
genre_ids = torch.tensor(
    pd.DataFrame(audio["spotify_genre_ids"].to_list()).values,
    dtype=torch.long
)  # (N, MAX_GENRES)

In [37]:
class GenreEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

    def forward(self, genre_ids):
        """
        genre_ids: (batch, MAX_GENRES) long
        returns: (batch, embed_dim) float
        """
        e = self.emb(genre_ids)  # (batch, MAX_GENRES, embed_dim)

        # Mask out PAD tokens so they don't affect the pooled embedding
        mask = (genre_ids != self.pad_idx).unsqueeze(-1).float()  # (batch, MAX_GENRES, 1)
        e = e * mask

        denom = mask.sum(dim=1).clamp(min=1.0)  # (batch, 1)
        pooled = e.sum(dim=1) / denom          # (batch, embed_dim)
        return pooled

# Initialize encoder using your vocab size and PAD index
genre_encoder = GenreEncoder(vocab_size=vocab_size, embed_dim=16, pad_idx=genre2idx[PAD])

# Encode genres into vectors
genre_vecs = genre_encoder(genre_ids)  # (N, 16)

genre_vecs.shape

torch.Size([24186, 16])

In [38]:
# embedding dim (16, but can be adjusted)
embed_dim = genre_vecs.shape[1]

# convert the genre embedding tensor back to a pandas DataFrame
genre_vec_df = pd.DataFrame(
    # chat told me to add this in case anyone using GPU
    genre_vecs.detach().cpu().numpy(),
    columns=[f"genre_emb_{i+1}" for i in range(embed_dim)],
    index=audio.index
)

# preview first few rows of the genre embedding
genre_vec_df.head(5)

,genre_emb_1,genre_emb_2,genre_emb_3,genre_emb_4,genre_emb_5,genre_emb_6,genre_emb_7,genre_emb_8,genre_emb_9,genre_emb_10,genre_emb_11,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16
2,-0.077386,-0.230027,-0.527311,-0.734082,0.346165,0.215216,-0.827345,0.426096,-0.108040,0.061576,0.026820,0.698124,-0.240986,-0.032105,-0.580920,0.184255
3,-0.967503,-0.240076,0.582942,-0.071300,0.525413,0.358620,1.083216,-1.112254,-0.841985,0.075509,0.342734,-0.476329,0.731314,-1.287699,0.409733,-0.277434
4,0.241397,0.478558,-1.104392,0.149362,-0.282572,0.440134,0.231554,0.876986,0.864115,0.333537,0.514130,-0.046675,-0.706932,-0.516953,0.516098,1.016299
5,0.043611,0.576857,-0.645123,0.271697,-1.029608,0.413935,0.205704,1.457733,0.621160,-0.456910,0.862064,-0.061422,-1.126436,-0.672843,-0.241420,0.633550
7,0.820644,-0.196929,-0.020945,-0.121095,-0.178029,0.032046,-0.247866,-1.042331,0.335073,0.344396,1.270917,-0.059303,0.264784,-0.558835,-0.254581,-0.913736


In [39]:
# attach embeddings to audio df
audio = pd.concat([audio, genre_vec_df], axis=1)

# drop old genre columns
### COMMENTED OUT FOR NEW MODELS ###
# audio = audio.drop(columns=["spotify_genre", "spotify_genre_ids"])

# columns
print(audio.columns)

Index(['SongID', 'spotify_genre', 'spotify_track_duration_ms',
       'spotify_track_explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'spotify_genre_ids',
       'genre_emb_1', 'genre_emb_2', 'genre_emb_3', 'genre_emb_4',
       'genre_emb_5', 'genre_emb_6', 'genre_emb_7', 'genre_emb_8',
       'genre_emb_9', 'genre_emb_10', 'genre_emb_11', 'genre_emb_12',
       'genre_emb_13', 'genre_emb_14', 'genre_emb_15', 'genre_emb_16'],
      dtype='object')


In [40]:
# merge the cleaned audio features with aggregated chart data using SongID
df = audio.merge(lb_song, on="SongID", how="inner")

# sort df by first week
df = df.sort_values(by="first_week")

df.columns

Index(['SongID', 'spotify_genre', 'spotify_track_duration_ms',
       'spotify_track_explicit', 'danceability', 'energy', 'key', 'loudness',
       'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness',
       'valence', 'tempo', 'time_signature', 'spotify_genre_ids',
       'genre_emb_1', 'genre_emb_2', 'genre_emb_3', 'genre_emb_4',
       'genre_emb_5', 'genre_emb_6', 'genre_emb_7', 'genre_emb_8',
       'genre_emb_9', 'genre_emb_10', 'genre_emb_11', 'genre_emb_12',
       'genre_emb_13', 'genre_emb_14', 'genre_emb_15', 'genre_emb_16',
       'first_week', 'last_week', 'peak_position', 'weeks_on_chart', 'hit'],
      dtype='object')

In [41]:
hit_percentage = df["hit"].mean() * 100
hit_percentage

np.float64(14.30225346289022)

In [42]:
# first week in the complete df
min_week = df["first_week"].min()

# last week in the complete df
max_week = df["last_week"].max()

print(f"Date range covered by the dataset: {min_week:%B %d, %Y} to {max_week:%B %d, %Y}")
print(f"Number of rows in the cleaned dataset: {len(df):,}")

Date range covered by the dataset: August 02, 1958 to May 29, 2021
Number of rows in the cleaned dataset: 24,185


In [43]:
# sort the cleaned dataset from earliest date to latest
df = df.sort_values(by="first_week")

In [44]:
# spot check begining
df.head(5)

,SongID,spotify_genre,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,...,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16,first_week,last_week,peak_position,weeks_on_chart,hit
10923,JudyFrankie Vaughan,[rock-and-roll],125933.0,0,0.517,0.420,9.0,-12.751,0.0,0.0713,...,1.435648,1.443909,-1.505699,2.318893,0.081307,1958-08-02,1958-08-02,100,1,0
343,A Certain SmileJohnny Mathis,"[adult standards, brill building pop, easy lis...",168293.0,0,0.233,0.337,5.0,-10.031,1.0,0.0307,...,0.706544,-0.243702,0.282202,-0.436747,-0.474178,1958-08-02,1958-09-27,22,9,0
10977,Just A DreamJimmy Clanton And His Rockets,[unknown],152973.0,0,0.610,0.326,7.0,-12.266,1.0,0.0505,...,-0.598570,1.153655,-2.776071,1.255487,0.059592,1958-08-02,1958-11-08,4,15,1
18936,Summertime BluesEddie Cochran,"[adult standards, brill building pop, rock-and...",119360.0,0,0.715,0.882,11.0,-8.610,0.0,0.0593,...,1.031384,0.602439,-0.855159,0.357241,0.683984,1958-08-02,1958-11-15,8,16,1
7949,High School ConfidentialJerry Lee Lewis And Hi...,[unknown],150026.0,0,0.608,0.920,10.0,-6.792,1.0,0.0422,...,-0.598570,1.153655,-2.776071,1.255487,0.059592,1958-08-02,1958-08-09,63,2,0


In [45]:
# spot check end
df.tail(5)

,SongID,spotify_genre,spotify_track_duration_ms,spotify_track_explicit,danceability,energy,key,loudness,mode,speechiness,...,genre_emb_12,genre_emb_13,genre_emb_14,genre_emb_15,genre_emb_16,first_week,last_week,peak_position,weeks_on_chart,hit
6220,FractionsNicki Minaj,"[dance pop, hip pop, pop, pop rap, post-teen p...",181690.0,1,0.924,0.523,0.0,-6.362,1.0,0.1320,...,-0.410984,-0.365711,0.298014,0.370077,0.318850,2021-05-29,2021-05-29,52,1,0
22943,White TeethYoungBoy Never Broke Again,"[baton rouge rap, trap]",174683.0,1,0.638,0.660,3.0,-7.115,0.0,0.2990,...,1.437849,-0.610061,1.225767,0.347951,-0.072817,2021-05-29,2021-05-29,78,1,0
18740,StraighteninMigos,"[atl hip hop, hip hop, pop rap, rap, southern ...",255532.0,1,0.847,0.629,9.0,-5.810,1.0,0.1020,...,0.047484,-0.248337,-0.250302,-0.004760,0.102077,2021-05-29,2021-05-29,38,1,0
20739,Things A Man Oughta KnowLainey Wilson,"[contemporary country, country pop]",203373.0,0,0.659,0.683,3.0,-5.623,1.0,0.0312,...,0.552892,-0.312803,1.158746,0.594203,0.791762,2021-05-29,2021-05-29,94,1,0
2708,Build A BitchBella Poarch,[unknown],122772.0,1,0.855,0.463,3.0,-7.454,1.0,0.0367,...,-0.598570,1.153655,-2.776071,1.255487,0.059592,2021-05-29,2021-05-29,58,1,0


In [46]:
# final spot check
df.isnull().sum()

,0
SongID,0
spotify_genre,0
spotify_track_duration_ms,0
spotify_track_explicit,0
danceability,0
energy,0
key,0
loudness,0
mode,0
speechiness,0


In [47]:
# Separate features and target
X = df.drop(columns=["SongID", "hit", "peak_position", "weeks_on_chart"])
y = df["hit"]

# First split: 70% train, 30% temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=426,
    stratify=y
)

# Second split: split the 30% temp into 15% val and 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,   # half of 30% = 15%
    random_state=426,
    stratify=y_temp
)

In [48]:
## Uncomment Genre drop for this

# ===============================
# FINAL DATA PREP FOR DEEP LEARNING
# ===============================

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score

# -----------------------
# Recreate X and y safely
# -----------------------

feature_drop_cols = [
    "SongID",
    "hit",
    "peak_position",
    "weeks_on_chart",
    "first_week",
    "last_week"
]

X = df.drop(columns=feature_drop_cols)
y = df["hit"]

# -----------------------
# Train / Val / Test Split
# -----------------------

from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=426,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=426,
    stratify=y_temp
)

# -----------------------
# Standardize Features
# -----------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# -----------------------
# Convert to Tensors
# -----------------------

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val_scaled, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor   = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

input_dim = X_train_tensor.shape[1]
print("Input dimension:", input_dim)

# ===============================
# MODEL DEFINITION
# ===============================

class HitPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)  # NO sigmoid here
        )

    def forward(self, x):
        return self.net(x)

model = HitPredictor(input_dim)

# ===============================
# HANDLE CLASS IMBALANCE
# ===============================

pos_weight = torch.tensor(
    [(len(y_train) - y_train.sum()) / y_train.sum()],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# ===============================
# TRAINING LOOP
# ===============================

epochs = 50
best_val_loss = float("inf")
patience = 5
counter = 0

for epoch in range(epochs):

    # ---- Training ----
    model.train()
    optimizer.zero_grad()

    train_logits = model(X_train_tensor)
    train_loss = criterion(train_logits, y_train_tensor)

    train_loss.backward()
    optimizer.step()

    # ---- Validation ----
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_tensor)
        val_loss = criterion(val_logits, y_val_tensor)

    # ---- Early Stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_model_state = model.state_dict()
    else:
        counter += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss.item():.4f}")
        print(f"Val Loss:   {val_loss.item():.4f}")
        print("-" * 30)

    if counter >= patience:
        print("Early stopping triggered.")
        break

# Load best model
model.load_state_dict(best_model_state)

# ===============================
# TEST EVALUATION
# ===============================

model.eval()
with torch.no_grad():
    test_logits = model(X_test_tensor)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs > 0.5).float()

accuracy = accuracy_score(y_test, test_preds.numpy())
auc = roc_auc_score(y_test, test_probs.numpy())

print("=================================")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test ROC-AUC:  {auc:.4f}")
print("=================================")

ValueError: setting an array element with a sequence.

In [ ]:
# =====================================
# IMPROVED MODEL: Features + Scheduler + Threshold Tuning
# =====================================

import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve
import numpy as np

# =====================================
# 1️⃣ FEATURE ENGINEERING (NEW)
# =====================================

df["energy_x_dance"] = df["energy"] * df["danceability"]
df["valence_x_energy"] = df["valence"] * df["energy"]
df["tempo_x_energy"] = df["tempo"] * df["energy"]
df["loudness_abs"] = df["loudness"].abs()

# =====================================
# BUILD FEATURES
# =====================================

feature_drop_cols = [
    "SongID",
    "hit",
    "peak_position",
    "weeks_on_chart",
    "first_week",
    "last_week"
]

X = df.drop(columns=feature_drop_cols)
y = df["hit"]

# Keep numeric columns only
X = X.select_dtypes(include=[np.number])

# =====================================
# TRAIN / VAL / TEST SPLIT
# =====================================

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=426, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    random_state=426, stratify=y_temp
)

# =====================================
# STANDARDIZE
# =====================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val, dtype=torch.float32)
X_test  = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val   = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

input_dim = X_train.shape[1]
print("New input dimension:", input_dim)

# =====================================
# 2️⃣ SLIGHTLY STRONGER MODEL
# =====================================

class HitPredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model = HitPredictor(input_dim)

# =====================================
# LOSS (IMBALANCE HANDLING)
# =====================================

pos_weight = torch.tensor(
    [(len(y_train) - y_train.sum()) / y_train.sum()],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 4️⃣ LEARNING RATE SCHEDULER (NEW)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=3,
    factor=0.5
)

# =====================================
# TRAINING
# =====================================

epochs = 80
best_val_loss = float("inf")
patience = 7
counter = 0

for epoch in range(epochs):

    model.train()
    optimizer.zero_grad()

    logits = model(X_train)
    loss = criterion(logits, y_train)

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = criterion(val_logits, y_val)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        best_state = model.state_dict()
    else:
        counter += 1

    if (epoch+1) % 5 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch+1}")
        print("Train loss:", round(loss.item(),4))
        print("Val loss:", round(val_loss.item(),4))
        print("LR:", current_lr)
        print("-"*30)

    if counter >= patience:
        print("Early stopping triggered")
        break

model.load_state_dict(best_state)

# =====================================
# TEST EVALUATION
# =====================================

model.eval()
with torch.no_grad():
    test_logits = model(X_test)
    test_probs = torch.sigmoid(test_logits).numpy().flatten()

# ROC-AUC
auc = roc_auc_score(y_test, test_probs)

# 5️⃣ OPTIMAL THRESHOLD (NEW)
fpr, tpr, thresholds = roc_curve(y_test, test_probs)
optimal_idx = (tpr - fpr).argmax()
optimal_threshold = thresholds[optimal_idx]

test_preds = (test_probs > optimal_threshold).astype(int)
accuracy = accuracy_score(y_test, test_preds)

print("=================================")
print("Test ROC-AUC:", round(auc,4))
print("Optimal Threshold:", round(optimal_threshold,4))
print("Test Accuracy (Optimal Threshold):", round(accuracy,4))
print("=================================")

In [51]:
# =====================================
# END-TO-END GENRE EMBEDDING MODEL  --> THIS IS MY BEST MODEL BY ROC-AUC
# =====================================

import torch
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

# -------------------------------------
# FEATURE ENGINEERING (keep improvements)
# -------------------------------------

df["energy_x_dance"] = df["energy"] * df["danceability"]
df["valence_x_energy"] = df["valence"] * df["energy"]
df["tempo_x_energy"] = df["tempo"] * df["energy"]
df["loudness_abs"] = df["loudness"].abs()

# -------------------------------------
# SPLIT FEATURES
# -------------------------------------

genre_ids = np.stack(df["spotify_genre_ids"].values)

feature_drop_cols = [
    "SongID",
    "hit",
    "peak_position",
    "weeks_on_chart",
    "first_week",
    "last_week",
    "spotify_genre_ids"
]

X_numeric = df.drop(columns=feature_drop_cols)
y = df["hit"].values

X_numeric = X_numeric.select_dtypes(include=[np.number]).values

# -------------------------------------
# TRAIN / VAL / TEST SPLIT
# -------------------------------------

(
    X_train_num,
    X_temp_num,
    genre_train,
    genre_temp,
    y_train,
    y_temp
) = train_test_split(
    X_numeric,
    genre_ids,
    y,
    test_size=0.30,
    random_state=426,
    stratify=y
)

(
    X_val_num,
    X_test_num,
    genre_val,
    genre_test,
    y_val,
    y_test
) = train_test_split(
    X_temp_num,
    genre_temp,
    y_temp,
    test_size=0.50,
    random_state=426,
    stratify=y_temp
)

# -------------------------------------
# SCALE NUMERIC FEATURES
# -------------------------------------

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_num)
X_val_num   = scaler.transform(X_val_num)
X_test_num  = scaler.transform(X_test_num)

# tensors
X_train_num = torch.tensor(X_train_num, dtype=torch.float32)
X_val_num   = torch.tensor(X_val_num, dtype=torch.float32)
X_test_num  = torch.tensor(X_test_num, dtype=torch.float32)

genre_train = torch.tensor(genre_train, dtype=torch.long)
genre_val   = torch.tensor(genre_val, dtype=torch.long)
genre_test  = torch.tensor(genre_test, dtype=torch.long)

y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_val   = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

numeric_dim = X_train_num.shape[1]

# -------------------------------------
# MODEL WITH TRAINABLE GENRE EMBEDDINGS
# -------------------------------------

class HitPredictor(nn.Module):
    def __init__(self, numeric_dim, vocab_size, embed_dim=16, pad_idx=0):
        super().__init__()

        # -------- Genre Embedding --------
        self.genre_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        # total input size after concat
        combined_dim = numeric_dim + embed_dim

        # -------- Main Network --------
        self.net = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, numeric_x, genre_ids):

        # genre_ids shape: (batch, num_genres_per_song)
        embeds = self.genre_embedding(genre_ids)

        # average pooling across genres
        embeds = embeds.mean(dim=1)

        # concatenate numeric + embedding
        x = torch.cat([numeric_x, embeds], dim=1)

        return self.net(x)

# initialize
model = HitPredictor(
    numeric_dim=numeric_dim,
    vocab_size=vocab_size,
    embed_dim=16,
    pad_idx=0
)

# -------------------------------------
# LOSS + OPTIMIZER
# -------------------------------------

pos_weight = torch.tensor(
    [(len(y_train) - y_train.sum()) / y_train.sum()]
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5
)

# -------------------------------------
# TRAINING
# -------------------------------------

best_val_loss = float("inf")
patience = 7
counter = 0

for epoch in range(80):

    model.train()
    optimizer.zero_grad()

    logits = model(X_train_num, genre_train)
    loss = criterion(logits, y_train)

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_num, genre_val)
        val_loss = criterion(val_logits, y_val)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}")
        print("Train:", round(loss.item(),4),
              " Val:", round(val_loss.item(),4))

    if counter >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_state)

# -------------------------------------
# EVALUATION
# -------------------------------------

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(
        model(X_test_num, genre_test)
    ).numpy().flatten()

auc = roc_auc_score(y_test, probs)

fpr, tpr, thresholds = roc_curve(y_test, probs)
opt_idx = (tpr - fpr).argmax()
opt_thresh = thresholds[opt_idx]

preds = (probs > opt_thresh).astype(int)
acc = accuracy_score(y_test, preds)

print("=================================")
print("ROC-AUC:", round(auc,4))
print("Optimal Threshold:", round(opt_thresh,4))
print("Accuracy:", round(acc,4))
print("=================================")

Epoch 5
Train: 1.1543  Val: 1.1736
Epoch 10
Train: 1.1138  Val: 1.1452
Epoch 15
Train: 1.0913  Val: 1.1139
Epoch 20
Train: 1.0744  Val: 1.0973
Epoch 25
Train: 1.06  Val: 1.0922
Epoch 30
Train: 1.0539  Val: 1.088
Epoch 35
Train: 1.0408  Val: 1.0834
Epoch 40
Train: 1.0348  Val: 1.0813
Epoch 45
Train: 1.0267  Val: 1.0805
Epoch 50
Train: 1.0113  Val: 1.0797
Epoch 55
Train: 1.013  Val: 1.0797
Early stopping
ROC-AUC: 0.7261
Optimal Threshold: 0.5502
Accuracy: 0.7111


In [53]:
# =====================================
# RESIDUAL GENRE EMBEDDING MODEL
# =====================================

import torch
import torch.nn as nn

# -------------------------------------
# Residual Block
# -------------------------------------

class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()

        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        # skip connection
        return x + self.block(x)


# -------------------------------------
# Main Model
# -------------------------------------

class HitPredictor(nn.Module):
    def __init__(self, numeric_dim, vocab_size, embed_dim=16, pad_idx=0):
        super().__init__()

        # ---- Genre Embedding ----
        self.genre_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_idx
        )

        combined_dim = numeric_dim + embed_dim

        # ---- Input Layer ----
        self.input_layer = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        # ---- Residual Blocks ----
        self.res1 = ResidualBlock(256, dropout=0.3)
        self.res2 = ResidualBlock(256, dropout=0.3)

        # ---- Prediction Head ----
        self.head = nn.Sequential(
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, numeric_x, genre_ids):

        # Embed genres
        embeds = self.genre_embedding(genre_ids)

        # average pooling across genres
        embeds = embeds.mean(dim=1)

        # concatenate numeric + embeddings
        x = torch.cat([numeric_x, embeds], dim=1)

        x = self.input_layer(x)
        x = self.res1(x)
        x = self.res2(x)

        return self.head(x)


# -------------------------------------
# Initialize Model
# -------------------------------------

# compute vocab size automatically
vocab_size = int(genre_train.max().item()) + 1

model = HitPredictor(
    numeric_dim=numeric_dim,
    vocab_size=vocab_size,
    embed_dim=16,
    pad_idx=0
)

print("Model initialized.")

# =====================================
# TRAINING + EVALUATION
# =====================================

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

# ----- Loss -----
pos_weight = torch.tensor(
    [(len(y_train) - y_train.sum()) / y_train.sum()],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5
)

# -------------------------------------
# TRAIN LOOP
# -------------------------------------

best_val_loss = float("inf")
patience = 7
counter = 0
epochs = 80

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    optimizer.zero_grad()

    logits = model(X_train_num, genre_train)
    loss = criterion(logits, y_train)

    loss.backward()

    # gradient clipping (important for residual nets)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    # ---- VALIDATION ----
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_num, genre_val)
        val_loss = criterion(val_logits, y_val)

    scheduler.step(val_loss)

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}")
        print("Train:", round(loss.item(),4),
              " Val:", round(val_loss.item(),4))

    if counter >= patience:
        print("Early stopping")
        break

# load best model
model.load_state_dict(best_state)

# -------------------------------------
# TEST EVALUATION
# -------------------------------------

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(
        model(X_test_num, genre_test)
    ).numpy().flatten()

auc = roc_auc_score(y_test, probs)

fpr, tpr, thresholds = roc_curve(y_test, probs)
opt_idx = (tpr - fpr).argmax()
opt_thresh = thresholds[opt_idx]

preds = (probs > opt_thresh).astype(int)
acc = accuracy_score(y_test, preds)

print("=================================")
print("ROC-AUC:", round(auc,4))
print("Optimal Threshold:", round(opt_thresh,4))
print("Accuracy:", round(acc,4))
print("=================================")

Model initialized.
Epoch 5
Train: 1.0944  Val: 1.1492
Epoch 10
Train: 1.0514  Val: 1.1174
Epoch 15
Train: 1.0183  Val: 1.0989
Epoch 20
Train: 0.9952  Val: 1.0844
Epoch 25
Train: 0.9651  Val: 1.0763
Epoch 30
Train: 0.9339  Val: 1.0825
Early stopping
ROC-AUC: 0.7151
Optimal Threshold: 0.5667
Accuracy: 0.7189


In [55]:
# =====================================
# WIDE & DEEP MODEL (BEST FOR TABULAR)
# =====================================

import torch
import torch.nn as nn

class HitPredictor(nn.Module):
    def __init__(self, numeric_dim, vocab_size, embed_dim=16, pad_idx=0):
        super().__init__()

        # ---- Genre Embedding ----
        self.genre_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=pad_idx
        )

        combined_dim = numeric_dim + embed_dim

        # -------------------------
        # WIDE (linear) component
        # -------------------------
        self.wide = nn.Linear(combined_dim, 1)

        # -------------------------
        # DEEP component
        # -------------------------
        self.deep = nn.Sequential(
            nn.Linear(combined_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),

            nn.Linear(64, 1)
        )

    def forward(self, numeric_x, genre_ids):

        embeds = self.genre_embedding(genre_ids)
        embeds = embeds.mean(dim=1)

        x = torch.cat([numeric_x, embeds], dim=1)

        wide_out = self.wide(x)
        deep_out = self.deep(x)

        # combine both paths
        return wide_out + deep_out

        vocab_size = int(genre_train.max().item()) + 1

model = HitPredictor(
    numeric_dim=numeric_dim,
    vocab_size=vocab_size
)

print("Wide & Deep model initialized.")

# =====================================
# TRAINING + EVALUATION
# =====================================

from sklearn.metrics import roc_auc_score, accuracy_score, roc_curve

# ----- Loss -----
pos_weight = torch.tensor(
    [(len(y_train) - y_train.sum()) / y_train.sum()],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", patience=3, factor=0.5
)

# -------------------------------------
# TRAIN LOOP
# -------------------------------------

best_val_loss = float("inf")
patience = 7
counter = 0
epochs = 80

for epoch in range(epochs):

    # ---- TRAIN ----
    model.train()
    optimizer.zero_grad()

    logits = model(X_train_num, genre_train)
    loss = criterion(logits, y_train)

    loss.backward()

    # gradient clipping (important for residual nets)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    optimizer.step()

    # ---- VALIDATION ----
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_num, genre_val)
        val_loss = criterion(val_logits, y_val)

    scheduler.step(val_loss)

    # early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = model.state_dict()
        counter = 0
    else:
        counter += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}")
        print("Train:", round(loss.item(),4),
              " Val:", round(val_loss.item(),4))

    if counter >= patience:
        print("Early stopping")
        break

# load best model
model.load_state_dict(best_state)

# -------------------------------------
# TEST EVALUATION
# -------------------------------------

model.eval()
with torch.no_grad():
    probs = torch.sigmoid(
        model(X_test_num, genre_test)
    ).numpy().flatten()

auc = roc_auc_score(y_test, probs)

fpr, tpr, thresholds = roc_curve(y_test, probs)
opt_idx = (tpr - fpr).argmax()
opt_thresh = thresholds[opt_idx]

preds = (probs > opt_thresh).astype(int)
acc = accuracy_score(y_test, preds)

print("=================================")
print("ROC-AUC:", round(auc,4))
print("Optimal Threshold:", round(opt_thresh,4))
print("Accuracy:", round(acc,4))
print("=================================")

Wide & Deep model initialized.
Epoch 5
Train: 1.1817  Val: 1.206
Epoch 10
Train: 1.1326  Val: 1.1646
Epoch 15
Train: 1.1055  Val: 1.1213
Epoch 20
Train: 1.0865  Val: 1.105
Epoch 25
Train: 1.0773  Val: 1.1018
Epoch 30
Train: 1.0608  Val: 1.095
Epoch 35
Train: 1.0527  Val: 1.0877
Epoch 40
Train: 1.0399  Val: 1.0837
Epoch 45
Train: 1.0322  Val: 1.0813
Epoch 50
Train: 1.0233  Val: 1.0791
Epoch 55
Train: 1.0171  Val: 1.077
Epoch 60
Train: 1.0173  Val: 1.0756
Epoch 65
Train: 0.9973  Val: 1.076
Early stopping
ROC-AUC: 0.7215
Optimal Threshold: 0.5232
Accuracy: 0.6805
